In [ ]:
import concurrent.futures
import csv
import json
import os
import sys
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize

plt.rcParams["axes.formatter.useoffset"] = False

root_dir = os.path.abspath("..")
if root_dir not in sys.path:
    sys.path.append(root_dir)

from current_setpoints.models.machines import ieee_machine2_trained_neural_flux
from current_setpoints.models.forward_model import ForwardModel, Fault, count_peaks_at_limit
from current_setpoints.optimization import FeasibilityOptimizer

CURR_MAX, VOLT_MAX = 30.0, 13.0
N_THETA_GRID = 90
SOLVER_OPTS = {"disp": False, "ftol": 1e-8, "maxiter": 300}

OUT_DIR = "../figs/diagnostics"
os.makedirs(OUT_DIR, exist_ok=True)

drive = ieee_machine2_trained_neural_flux(curr_max=CURR_MAX, volt_max=VOLT_MAX)
CONST_MECH_SPEED = 30.0 / (np.pi * drive.n_ppairs)

FAULT_MODES = {
    "healthy": (),
    "1-fault": (4,),
    "2-adj": (3, 4),
    "2-non-adj": (2, 4),
}
FWD_BY_MODE = {name: ForwardModel(drive, fault=Fault(op), n_theta=700) for name, op in FAULT_MODES.items()}
PHASE_LETTERS = ["a", "b", "c", "d", "e"]


def make_opt(mode, **kwargs):
    return FeasibilityOptimizer(FWD_BY_MODE[mode], SOLVER_OPTS, n_grid=N_THETA_GRID, **kwargs)


TOL = make_opt("healthy").tol

print(f"const_mech_speed = {CONST_MECH_SPEED:.4f}")
print(f"convergence tolerance TOL = {TOL:g} V  ({TOL / VOLT_MAX * 100:.4f}% of the {VOLT_MAX} V limit)")
print(f"outputs -> {os.path.abspath(OUT_DIR)}")

In [ ]:
def call_with_timeout(timeout, fn, *args, **kwargs):
    """Hard wall-clock cap. On Windows the underlying SLSQP call cannot be
    killed, so it keeps running in its thread -- but this notebook is never
    blocked past `timeout` by any single point."""
    executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    future = executor.submit(fn, *args, **kwargs)
    try:
        result = future.result(timeout=timeout)
        executor.shutdown(wait=False)
        return result, None
    except concurrent.futures.TimeoutError:
        executor.shutdown(wait=False)
        return None, "timeout"
    except Exception as exc:
        executor.shutdown(wait=False)
        return None, f"exception: {exc!r}"


def write_csv(path, rows, fields):
    """Rewrite a CSV from scratch. Callers append to their list and re-call, so
    a run interrupted midway still leaves everything gathered so far on disk."""
    if not rows:
        return
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)


def r0_probe(opt, omega, maps):
    """R0's own per-node torque ceiling. R0 solves each angle independently and
    IGNORES the omega*L*di/dtheta coupling, so this ceiling is an OVERESTIMATE
    of what is dynamically reachable -- that gap is what most of this notebook
    measures. Non-positive means the machine cannot make positive torque at
    this speed under this fault at all."""
    r0_grid, ok = opt._run_per_angle(omega, maps, None, maximize=True)
    if not np.any(ok):
        return float("nan"), r0_grid, ok
    tq = np.array([opt.fwd.drive.torque(omega, r0_grid[i]) if ok[i] else np.nan for i in range(opt.n_grid)])
    return float(np.nanmin(tq)), r0_grid, ok


def analyze_point(mode, rpm, frac=None, torque=None, timeout=900.0, verbose=True, instrument=False):
    """Full characterisation of ONE setpoint: where R0 starts, what the solver
    reaches, and -- when it fails -- BY HOW MUCH it misses TOL.

    Everything reported is re-derived from the returned trajectory with
    voltage_residuals_exact rather than read off sol.success, so a green flag
    that is not independently verifiable cannot slip through.

    instrument=True also times every internal trust-region solve, giving the
    per-iteration cost breakdown (which mu was tried, how long each took)."""
    fwd = FWD_BY_MODE[mode]
    opt = make_opt(mode)
    omega = rpm / CONST_MECH_SPEED
    maps = opt._precompute_maps(omega)
    idx_grid = maps["idx_grid"]

    t_max_r0, _, _ = r0_probe(opt, omega, maps)
    if torque is None:
        torque = (frac if frac is not None else 0.8) * t_max_r0
    frac_eff = torque / t_max_r0 if np.isfinite(t_max_r0) and t_max_r0 != 0 else np.nan

    row = {"mode": mode, "rpm": rpm, "torque": torque, "frac_of_R0_ceiling": frac_eff,
           "R0_torque_ceiling": t_max_r0}
    calls = []

    if not np.isfinite(t_max_r0) or t_max_r0 <= 0:
        row.update(outcome="physically infeasible (R0 ceiling <= 0)", success=False,
                   verified_feasible=False, final_resid=np.nan, miss=np.nan, seconds=0.0)
        if verbose:
            print(f"  {mode} @ {rpm} RPM: R0 ceiling {t_max_r0:.3f} Nm <= 0 -- no positive torque possible")
        return row, [], calls

    X0, ok0 = opt._run_per_angle(omega, maps, torque)
    if bool(np.all(ok0)):
        r0_res, _ = opt.voltage_residuals_exact(omega, X0, idx_grid)
        row["R0_max_resid"] = float(np.max(r0_res))
        row["R0_n_violating"] = int(np.sum(r0_res > TOL))
    else:
        row["R0_max_resid"] = np.nan
        row["R0_n_violating"] = -1
        row["R0_failed_nodes"] = int(np.sum(~ok0))

    if instrument:
        original = opt._solve_joint_proximal

        def timed(maps_, om, tt, X_init, active, mu):
            t0 = time.perf_counter()
            out = original(maps_, om, tt, X_init, active, mu)
            calls.append({"mode": mode, "rpm": rpm, "torque": torque, "mu": mu,
                          "seconds": time.perf_counter() - t0, "lin_slack": out[1],
                          "solve_ok": out[2], "step_norm": float(np.linalg.norm(out[0] - X_init)),
                          "n_active": len(active)})
            return out

        opt._solve_joint_proximal = timed

    t0 = time.perf_counter()
    sol, err = call_with_timeout(timeout, opt.minimize_current, torque, omega)
    row["seconds"] = time.perf_counter() - t0

    if sol is None:
        row.update(outcome=err or "no solution", success=False, verified_feasible=False,
                   final_resid=np.nan, miss=np.nan)
        if verbose:
            print(f"  {mode} @ {rpm} RPM T={torque:.3f}: {err} ({row['seconds']:.0f}s)")
        return row, [], calls

    hist = sol.diagnostics.get("history", [])
    iters = [h for h in hist if isinstance(h.get("iter"), int) and h["iter"] >= 0]
    row["n_iters"] = len(iters)
    row["last_action"] = hist[-1].get("action", "") if hist else ""
    row["fail_reason"] = sol.diagnostics.get("fail_reason", "")

    if np.all(np.isfinite(sol.curr_dq)):
        r_fin, _ = opt.voltage_residuals_exact(omega, sol.curr_dq, idx_grid)
        tq_fin = np.array([fwd.drive.torque(omega, sol.curr_dq[n]) for n in range(opt.n_grid)])
        final_resid = float(np.max(r_fin))
        row["final_resid"] = final_resid
        row["final_n_violating"] = int(np.sum(r_fin > TOL))
        row["miss"] = final_resid - TOL
        row["resid_pct_of_limit"] = final_resid / VOLT_MAX * 100.0
        row["torque_err"] = float(np.max(np.abs(tq_fin - torque)))
        row["peak_curr"] = float(np.max(np.abs(sol.curr_dq)))
        row["curr_headroom_pct"] = (1.0 - row["peak_curr"] / CURR_MAX) * 100.0
        row["verified_feasible"] = bool(final_resid <= TOL)
        if len(iters) >= 2:
            row["last_improvement"] = float(iters[-2]["max_V_residual"] - iters[-1]["max_V_residual"])
        else:
            row["last_improvement"] = np.nan
    else:
        row.update(final_resid=np.nan, miss=np.nan, verified_feasible=False)

    row["success"] = bool(sol.success)
    row["outcome"] = ("converged" if sol.success else
                      (row["fail_reason"] or row["last_action"] or "did not converge"))

    hist_rows = [{"mode": mode, "rpm": rpm, "torque": torque, "iter": h.get("iter"),
                  "max_V_residual": h.get("max_V_residual"), "n_violating": h.get("n_violating"),
                  "active_size": h.get("active_size"), "mu": h.get("mu"), "action": h.get("action")}
                 for h in hist]

    if verbose:
        flag = "OK " if row.get("verified_feasible") else "MISS"
        print(f"  [{flag}] {mode:10s} {rpm:6.0f} RPM  T={torque:6.3f}  "
              f"R0 {row.get('R0_max_resid', float('nan')):+7.3f} V -> final {row.get('final_resid', float('nan')):+8.4f} V  "
              f"(miss {row.get('miss', float('nan')):+.4f} V, resid {row.get('resid_pct_of_limit', float('nan')):+.3f}% of limit)  "
              f"iters={row.get('n_iters', 0)}  {row['outcome']}  ({row['seconds']:.0f}s)")
    return row, hist_rows, calls


DETAIL_FIELDS = ["mode", "rpm", "torque", "frac_of_R0_ceiling", "R0_torque_ceiling",
                 "R0_max_resid", "R0_n_violating", "R0_failed_nodes", "success", "verified_feasible",
                 "final_resid", "miss", "resid_pct_of_limit", "final_n_violating",
                 "n_iters", "last_improvement", "last_action", "outcome", "fail_reason",
                 "torque_err", "peak_curr", "curr_headroom_pct", "seconds"]
HISTORY_FIELDS = ["mode", "rpm", "torque", "iter", "max_V_residual", "n_violating",
                  "active_size", "mu", "action"]
CALL_FIELDS = ["mode", "rpm", "torque", "mu", "seconds", "lin_slack", "solve_ok", "step_norm", "n_active"]
print("helpers ready")

In [ ]:
BEMF_RPMS = [0.0, 200.0, 400.0, 600.0, 800.0, 1000.0, 1200.0, 1296.0, 1400.0, 1500.0, 1650.0, 1800.0]

bemf_rows = []
for mode in FAULT_MODES:
    fwd = FWD_BY_MODE[mode]
    kept = list(fwd.fault._kept)
    n_theta = fwd.vec_theta.size - 1
    theta_grid = np.linspace(0.0, 2 * np.pi, N_THETA_GRID, endpoint=False)
    idx_grid = np.round(theta_grid / (2 * np.pi) * n_theta).astype(int) % n_theta
    zero_curr = np.zeros(drive.dim)
    for rpm in BEMF_RPMS:
        omega = rpm / CONST_MECH_SPEED
        mags = np.array([np.abs(fwd.volt_map_at_theta(int(i), omega, zero_curr)[2][kept]) for i in idx_grid])
        peak = float(np.max(mags))
        bemf_rows.append({"mode": mode, "rpm": rpm, "peak_bemf": peak, "volt_max": VOLT_MAX,
                          "exceeds_limit": peak > VOLT_MAX,
                          "headroom_V": VOLT_MAX - peak})

write_csv(os.path.join(OUT_DIR, "bemf_headroom.csv"), bemf_rows,
          ["mode", "rpm", "peak_bemf", "volt_max", "exceeds_limit", "headroom_V"])

fig, ax = plt.subplots(figsize=(8.5, 5.5))
for mode in FAULT_MODES:
    pts = [(r["rpm"], r["peak_bemf"]) for r in bemf_rows if r["mode"] == mode]
    ax.plot([p[0] for p in pts], [p[1] for p in pts], marker="o", label=mode)
ax.axhline(VOLT_MAX, color="k", ls="--", lw=1.2, label=f"volt_max = {VOLT_MAX} V")
ax.set_xlabel("speed [RPM]"); ax.set_ylabel("peak open-circuit back-EMF [V]")
ax.set_title("Back-EMF vs voltage limit (crossing = flux-weakening required)")
ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "bemf_headroom.png"), dpi=110)
plt.close(fig)

for mode in FAULT_MODES:
    over = [r["rpm"] for r in bemf_rows if r["mode"] == mode and r["exceeds_limit"]]
    print(f"  {mode:11s} back-EMF exceeds {VOLT_MAX} V from {min(over):.0f} RPM" if over
          else f"  {mode:11s} back-EMF stays under {VOLT_MAX} V over the whole range")
print("saved bemf_headroom.csv / .png")

In [ ]:
LANDSCAPE_RPMS = [0.0, 400.0, 800.0, 1000.0, 1200.0, 1296.0, 1400.0, 1500.0, 1650.0, 1800.0]
LANDSCAPE_FRACS = [0.5, 0.8, 0.95]
LANDSCAPE_MODES = list(FAULT_MODES)

landscape_rows = []
LANDSCAPE_CSV = os.path.join(OUT_DIR, "r0_landscape.csv")
LANDSCAPE_FIELDS = ["mode", "rpm", "R0_torque_ceiling", "frac", "torque",
                    "n_violating", "n_nodes", "max_resid", "R0_all_nodes_ok"]

for mode in LANDSCAPE_MODES:
    opt = make_opt(mode)
    print(f"=== {mode} ===", flush=True)
    for rpm in LANDSCAPE_RPMS:
        omega = rpm / CONST_MECH_SPEED
        maps = opt._precompute_maps(omega)
        idx_grid = maps["idx_grid"]
        t_max_r0, _, _ = r0_probe(opt, omega, maps)
        if not np.isfinite(t_max_r0) or t_max_r0 <= 0:
            landscape_rows.append({"mode": mode, "rpm": rpm, "R0_torque_ceiling": t_max_r0,
                                   "frac": np.nan, "torque": np.nan, "n_violating": -1,
                                   "n_nodes": N_THETA_GRID, "max_resid": np.nan, "R0_all_nodes_ok": False})
            print(f"  {rpm:6.0f} RPM  ceiling {t_max_r0:7.3f} Nm <= 0, no positive torque possible", flush=True)
            write_csv(LANDSCAPE_CSV, landscape_rows, LANDSCAPE_FIELDS)
            continue
        for frac in LANDSCAPE_FRACS:
            torque = frac * t_max_r0
            X0, ok0 = opt._run_per_angle(omega, maps, torque)
            all_ok = bool(np.all(ok0))
            if all_ok:
                res, _ = opt.voltage_residuals_exact(omega, X0, idx_grid)
                n_viol, max_res = int(np.sum(res > TOL)), float(np.max(res))
            else:
                n_viol, max_res = -1, np.nan
            landscape_rows.append({"mode": mode, "rpm": rpm, "R0_torque_ceiling": t_max_r0,
                                   "frac": frac, "torque": torque, "n_violating": n_viol,
                                   "n_nodes": N_THETA_GRID, "max_resid": max_res, "R0_all_nodes_ok": all_ok})
            print(f"  {rpm:6.0f} RPM  ceiling {t_max_r0:6.3f}  frac {frac:.2f}  "
                  f"violating {n_viol:3d}/{N_THETA_GRID}  max_resid {max_res:+8.3f} V", flush=True)
            write_csv(LANDSCAPE_CSV, landscape_rows, LANDSCAPE_FIELDS)

print(f"\nsaved {LANDSCAPE_CSV}  ({len(landscape_rows)} rows)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

ax = axes[0]
for mode in LANDSCAPE_MODES:
    pts = sorted({(r["rpm"], r["R0_torque_ceiling"]) for r in landscape_rows if r["mode"] == mode})
    if pts:
        ax.plot([p[0] for p in pts], [p[1] for p in pts], marker="o", label=mode)
ax.axhline(0.0, color="k", ls="--", lw=1)
ax.set_xlabel("speed [RPM]"); ax.set_ylabel("R0 torque ceiling [Nm]")
ax.set_title("Torque ceiling vs speed (R0, static -- an OVERESTIMATE)")
ax.grid(alpha=0.3); ax.legend(fontsize=8)

ax = axes[1]
for mode in LANDSCAPE_MODES:
    for frac, ls in zip(LANDSCAPE_FRACS, ("-", "--", ":")):
        pts = sorted((r["rpm"], r["n_violating"]) for r in landscape_rows
                     if r["mode"] == mode and r.get("frac") == frac and r["n_violating"] >= 0)
        if pts:
            ax.plot([p[0] for p in pts], [p[1] for p in pts], ls=ls, marker=".", label=f"{mode} f={frac}")
ax.set_xlabel("speed [RPM]"); ax.set_ylabel(f"nodes violating (of {N_THETA_GRID})")
ax.set_title("Trajectory R0 leaves dynamically infeasible")
ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "r0_landscape.png"), dpi=110)
plt.close(fig)
print("saved r0_landscape.png")

In [ ]:
NODE_MODE, NODE_RPM, NODE_FRAC = "1-fault", 1200.0, 0.8

opt = make_opt(NODE_MODE)
omega = NODE_RPM / CONST_MECH_SPEED
maps = opt._precompute_maps(omega)
idx_grid = maps["idx_grid"]
t_max_r0, _, _ = r0_probe(opt, omega, maps)
torque = NODE_FRAC * t_max_r0
print(f"{NODE_MODE} @ {NODE_RPM} RPM, ceiling {t_max_r0:.3f} Nm, target {torque:.3f} Nm", flush=True)

node_rows = []
seed = opt.fwd.drive.seeds()[0]
from current_setpoints.optimization.optimizer import _run_slsqp
for n in range(N_THETA_GRID):
    cons = opt._per_angle_constraints(maps, omega, n, torque)
    t0 = time.perf_counter()
    best_x, _, ok = _run_slsqp(lambda x: float(np.sum(x ** 2)), cons, opt.fwd.drive.seeds(seed), opt.opts)
    node_rows.append({"node": n, "theta_deg": np.rad2deg(maps["theta_grid"][n]),
                      "ok": bool(ok), "seconds": time.perf_counter() - t0,
                      "curr_norm": float(np.linalg.norm(best_x)) if ok else np.nan})
    if ok:
        seed = best_x

X0, ok0 = opt._run_per_angle(omega, maps, torque)
if bool(np.all(ok0)):
    res, _ = opt.voltage_residuals_exact(omega, X0, idx_grid)
    for n in range(N_THETA_GRID):
        node_rows[n]["R0_resid"] = float(res[n])
        node_rows[n]["violating"] = bool(res[n] > TOL)

write_csv(os.path.join(OUT_DIR, f"per_node_{NODE_MODE}_{NODE_RPM:.0f}rpm.csv"), node_rows,
          ["node", "theta_deg", "ok", "seconds", "curr_norm", "R0_resid", "violating"])

n_fail = sum(1 for r in node_rows if not r["ok"])
slow = [r for r in node_rows if r["seconds"] > 1.0]
print(f"  nodes failing to solve: {n_fail}/{N_THETA_GRID}")
print(f"  nodes slower than 1 s : {len(slow)}  {[r['node'] for r in slow][:12]}")

if "R0_resid" in node_rows[0]:
    fig, ax = plt.subplots(figsize=(10, 4.5))
    th = [r["theta_deg"] for r in node_rows]
    rs = [r["R0_resid"] for r in node_rows]
    ax.plot(th, rs, marker=".", color="tab:blue")
    ax.axhline(0.0, color="k", ls="--", lw=1, label="voltage limit")
    ax.fill_between(th, 0, rs, where=[v > 0 for v in rs], color="tab:red", alpha=0.3, label="violating")
    ax.set_xlabel(r"$\theta$ [deg]"); ax.set_ylabel("R0 dynamic residual [V]")
    ax.set_title(f"{NODE_MODE} @ {NODE_RPM:.0f} RPM: where R0 violates around the period")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
    plt.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"per_node_{NODE_MODE}_{NODE_RPM:.0f}rpm.png"), dpi=110)
    plt.close(fig)
    print("  saved per-node CSV + plot")

In [ ]:
POINT_TIMEOUT = 900.0
INSTRUMENT = True

POINTS = [
    ("healthy",   1200.0, 0.8),
    ("healthy",   1500.0, 0.8),
    ("2-adj",     1200.0, 0.8),
    ("1-fault",   1200.0, 0.35),
    ("1-fault",   1200.0, 0.8),
]

DETAIL_CSV = os.path.join(OUT_DIR, "point_detail.csv")
HISTORY_CSV = os.path.join(OUT_DIR, "point_history.csv")
CALLS_CSV = os.path.join(OUT_DIR, "point_solve_calls.csv")

detail_rows, history_rows, call_rows = [], [], []
for mode, rpm, frac in POINTS:
    row, hrows, crows = analyze_point(mode, rpm, frac=frac, timeout=POINT_TIMEOUT, instrument=INSTRUMENT)
    detail_rows.append(row)
    history_rows.extend(hrows)
    call_rows.extend(crows)
    write_csv(DETAIL_CSV, detail_rows, DETAIL_FIELDS)
    write_csv(HISTORY_CSV, history_rows, HISTORY_FIELDS)
    write_csv(CALLS_CSV, call_rows, CALL_FIELDS)

print(f"\nsaved {DETAIL_CSV}, {HISTORY_CSV}, {CALLS_CSV}")

In [ ]:
hdr = (f"{'mode':11s} {'rpm':>6s} {'torque':>7s} {'R0 resid':>9s} {'final':>9s} {'miss':>9s} "
       f"{'resid%':>8s} {'iters':>5s} {'lastimp':>9s} {'Ihead%':>7s} {'outcome':<24s} {'secs':>6s}")
print(hdr)
print("-" * len(hdr))
for r in detail_rows:
    print(f"{r['mode']:11s} {r['rpm']:6.0f} {r.get('torque', float('nan')):7.3f} "
          f"{r.get('R0_max_resid', float('nan')):+9.3f} {r.get('final_resid', float('nan')):+9.4f} "
          f"{r.get('miss', float('nan')):+9.4f} {r.get('resid_pct_of_limit', float('nan')):+8.3f} "
          f"{r.get('n_iters', 0):5d} {r.get('last_improvement', float('nan')):9.4f} "
          f"{r.get('curr_headroom_pct', float('nan')):7.1f} {str(r.get('outcome', ''))[:24]:<24s} "
          f"{r.get('seconds', 0):6.0f}")

n_ok = sum(1 for r in detail_rows if r.get("verified_feasible"))
print(f"\n{n_ok}/{len(detail_rows)} points verified feasible (exact residual <= TOL = {TOL:g} V)")
print("miss     = final residual - TOL, in volts. Negative means inside tolerance.")
print("lastimp  = residual drop on the FINAL iteration. ~0 means it stalled;")
print("           a healthy value means it was still descending when it stopped.")
print("Ihead%   = unused current margin. High headroom on a failing point means")
print("           it is voltage-bound, not current-bound.")
print()
print("A miss far below the model's own fidelity (e.g. <0.01 V on a 13 V limit)")
print("is a FORMAL failure, not a physical one -- worth separating from the rest.")
for r in detail_rows:
    if not r.get("verified_feasible") and np.isfinite(r.get("miss", np.nan)):
        scale = ("below model fidelity" if r["final_resid"] < 0.01 else
                 "small but real" if r["final_resid"] < 0.5 else "large")
        print(f"  {r['mode']:10s} {r['rpm']:6.0f} RPM T={r['torque']:.3f}: "
              f"misses by {r['miss']:+.4f} V  ({scale})")

In [ ]:
fails = [r for r in detail_rows if not r.get("verified_feasible") and np.isfinite(r.get("miss", np.nan))]
fig, axes = plt.subplots(1, 3, figsize=(19, 5.2))

ax = axes[0]
if fails:
    labels = [f"{r['mode']}\n{r['rpm']:.0f}rpm T={r['torque']:.2f}" for r in fails]
    ax.bar(range(len(fails)), [r["final_resid"] for r in fails], color="tab:red", edgecolor="black", linewidth=0.5)
    ax.set_xticks(range(len(fails)))
    ax.set_xticklabels(labels, fontsize=7, rotation=30, ha="right")
    ax.set_yscale("log")
    ax.axhline(TOL, color="k", ls="--", lw=1.2, label=f"TOL = {TOL:g} V")
    ax.axhline(0.01, color="tab:orange", ls=":", lw=1.2, label="0.01 V (~0.08% of limit)")
    ax.axhline(0.1, color="tab:purple", ls=":", lw=1.2, label="0.1 V (~0.8% of limit)")
    ax.set_ylabel("final residual [V, log]")
    ax.set_title("How far each failing point misses")
    ax.legend(fontsize=7)
else:
    ax.text(0.5, 0.5, "no failing points", ha="center", va="center", transform=ax.transAxes)
ax.grid(alpha=0.3, axis="y")

ax = axes[1]
keys = sorted({(h["mode"], h["rpm"], round(h["torque"], 4)) for h in history_rows})
for mode, rpm, torque in keys:
    seq = [h for h in history_rows if h["mode"] == mode and h["rpm"] == rpm
           and round(h["torque"], 4) == torque and h.get("iter") is not None and h["iter"] >= 0
           and h.get("max_V_residual") is not None]
    if seq:
        ax.plot([h["iter"] for h in seq], [h["max_V_residual"] for h in seq],
                marker="o", ms=3, label=f"{mode} {rpm:.0f} T={torque:.2f}")
ax.axhline(TOL, color="k", ls="--", lw=1, label=f"TOL = {TOL:g} V")
ax.set_yscale("log")
ax.set_xlabel("outer iteration"); ax.set_ylabel("max exact residual [V]")
ax.set_title("Convergence (flat tail = stalled, not iteration-capped)")
ax.grid(alpha=0.3); ax.legend(fontsize=6)

ax = axes[2]
for mode, rpm, torque in keys:
    seq = [h for h in history_rows if h["mode"] == mode and h["rpm"] == rpm
           and round(h["torque"], 4) == torque and h.get("iter") is not None and h["iter"] >= 0
           and h.get("active_size") is not None]
    if seq:
        ax.plot([h["iter"] for h in seq], [h["active_size"] for h in seq],
                marker=".", label=f"{mode} {rpm:.0f}")
ax.set_xlabel("outer iteration"); ax.set_ylabel(f"active nodes (of {N_THETA_GRID})")
ax.set_title("Active-set growth (how much of the period is being corrected)")
ax.grid(alpha=0.3); ax.legend(fontsize=6)

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "point_detail.png"), dpi=110)
plt.close(fig)
print("saved point_detail.png")

if call_rows:
    print(f"\ntrust-region solve cost by mu ({len(call_rows)} solves):")
    for mu in sorted({c["mu"] for c in call_rows}):
        sub = [c for c in call_rows if c["mu"] == mu]
        print(f"  mu={mu:<8g} n={len(sub):3d}  mean {np.mean([c['seconds'] for c in sub]):6.1f}s  "
              f"mean step {np.mean([c['step_norm'] for c in sub]):8.3f}")

In [ ]:
TR_MODE, TR_RPM, TR_FRAC = "healthy", 1500.0, 0.8
TR_MUS = [0.0, 0.001, 0.01, 0.1, 1.0, 10.0]

opt = make_opt(TR_MODE)
omega = TR_RPM / CONST_MECH_SPEED
maps = opt._precompute_maps(omega)
idx_grid = maps["idx_grid"]
t_max_r0, _, _ = r0_probe(opt, omega, maps)
torque = TR_FRAC * t_max_r0

X, _ = opt._run_per_angle(omega, maps, torque)
r, _ = opt.voltage_residuals_exact(omega, X, idx_grid)
X_s, ok_s = opt._smoothed_warm_start(X, maps, omega, torque)
if ok_s:
    r_s, _ = opt.voltage_residuals_exact(omega, X_s, idx_grid)
    if np.max(r_s) < np.max(r):
        X, r = X_s, r_s
best = float(np.max(r))
violating = frozenset(int(n) for n in np.where(r > TOL)[0])
print(f"{TR_MODE} @ {TR_RPM} RPM T={torque:.3f}: start {best:+.4f} V, {len(violating)} active nodes\n", flush=True)

if not violating:
    print("WARNING: this point is ALREADY FEASIBLE, so the sweep below is degenerate.")
    print("         Choose a violating point (higher TR_RPM or TR_FRAC) to learn anything.")

maps_lin = opt._precompute_maps(omega, lin_curr=X)
tr_rows = []
print(f"{'mu':>8} | {'||dX||':>9} | {'lin claims':>10} | {'EXACT':>10} | {'gap':>9} | {'dT':>9} | {'secs':>6}", flush=True)
for mu in TR_MUS:
    t0 = time.perf_counter()
    Xc, s_lin, ok = opt._solve_joint_proximal(maps_lin, omega, torque, X, set(violating), mu)
    dt = time.perf_counter() - t0
    rc, _ = opt.voltage_residuals_exact(omega, Xc, idx_grid)
    tq = np.array([opt.fwd.drive.torque(omega, Xc[n]) for n in range(N_THETA_GRID)])
    exact = float(np.max(rc))
    tr_rows.append({"mode": TR_MODE, "rpm": TR_RPM, "torque": torque, "mu": mu,
                    "step_norm": float(np.linalg.norm(Xc - X)), "lin_slack": s_lin,
                    "exact_resid": exact, "linearisation_gap": exact - s_lin,
                    "torque_err": float(np.max(np.abs(tq - torque))), "improves": exact < best,
                    "solve_ok": ok, "seconds": dt})
    mark = "  <-- IMPROVES" if exact < best else ""
    print(f"{mu:8g} | {tr_rows[-1]['step_norm']:9.3f} | {s_lin:10.3f} | {exact:+10.4f} | "
          f"{tr_rows[-1]['linearisation_gap']:+9.3f} | {tr_rows[-1]['torque_err']:9.2e} | {dt:6.1f}{mark}", flush=True)

write_csv(os.path.join(OUT_DIR, "trust_region_sweep.csv"), tr_rows,
          ["mode", "rpm", "torque", "mu", "step_norm", "lin_slack", "exact_resid",
           "linearisation_gap", "torque_err", "improves", "solve_ok", "seconds"])

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot([r["step_norm"] for r in tr_rows], [r["exact_resid"] for r in tr_rows],
        marker="o", label="exact residual")
ax.plot([r["step_norm"] for r in tr_rows], [r["lin_slack"] for r in tr_rows],
        marker="s", ls="--", label="what the linear model claims")
ax.axhline(best, color="k", ls=":", lw=1.2, label=f"start = {best:.3f} V")
for r in tr_rows:
    ax.annotate(f"mu={r['mu']:g}", (r["step_norm"], r["exact_resid"]), fontsize=7,
                textcoords="offset points", xytext=(4, 5))
ax.set_xlabel("step length ||dX||"); ax.set_ylabel("residual [V]")
ax.set_title("Linearisation validity: claimed vs actual violation vs step length")
ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "trust_region_sweep.png"), dpi=110)
plt.close(fig)
print("\nsaved trust_region_sweep.csv / .png")
print("A big positive 'gap' means the linear model is confidently wrong at that step length.")

In [ ]:
MS_CASES = [("1-fault", 1650.0), ("2-adj", 1800.0)]
N_STARTS = 200

ms_rows = []
rng = np.random.default_rng(0)
for mode, rpm in MS_CASES:
    opt = make_opt(mode)
    omega = rpm / CONST_MECH_SPEED
    maps = opt._precompute_maps(omega)
    r0_grid, ok = opt._run_per_angle(omega, maps, None, maximize=True)
    tq = np.array([opt.fwd.drive.torque(omega, r0_grid[i]) if ok[i] else np.nan for i in range(opt.n_grid)])
    worst = int(np.nanargmin(tq))
    baseline = float(tq[worst])
    print(f"{mode} @ {rpm} RPM: worst node {worst}, built-in best torque {baseline:+.4f} Nm", flush=True)

    cons = opt._per_angle_constraints(maps, omega, worst, None)
    obj = lambda x: -opt.fwd.drive.torque(omega, x)
    dim = opt.fwd.drive.dim
    best_val, n_better = baseline, 0
    best_x = r0_grid[worst]
    t0 = time.perf_counter()
    for _ in range(N_STARTS):
        while True:
            x0 = rng.uniform(-CURR_MAX, CURR_MAX, size=dim)
            if np.linalg.norm(x0) <= CURR_MAX:
                break
        res = minimize(obj, x0, method="SLSQP", constraints=cons,
                       bounds=[(-1e4, 1e4)] * dim, options=SOLVER_OPTS)
        if res.success and -res.fun > best_val + 1e-6:
            best_val, best_x, n_better = float(-res.fun), res.x.copy(), n_better + 1
    dt = time.perf_counter() - t0

    peak = float(np.max(np.abs(best_x)))
    ms_rows.append({"mode": mode, "rpm": rpm, "worst_node": worst, "builtin_torque": baseline,
                    "multistart_best": best_val, "n_improvements": n_better, "n_starts": N_STARTS,
                    "peak_curr": peak, "curr_used_pct": peak / CURR_MAX * 100.0,
                    "verdict": "local-optimum artifact" if n_better else "genuine ceiling",
                    "seconds": dt})
    print(f"  {N_STARTS} random starts -> best {best_val:+.4f} Nm, improvements: {n_better}", flush=True)
    print(f"  peak |i| = {peak:.2f} A of {CURR_MAX} A ({peak / CURR_MAX * 100:.0f}% used) -> "
          f"{'CURRENT-bound' if peak > 0.97 * CURR_MAX else 'VOLTAGE-bound (current to spare)'}", flush=True)
    print(f"  verdict: {ms_rows[-1]['verdict']}  ({dt:.0f}s)\n", flush=True)

write_csv(os.path.join(OUT_DIR, "worst_node_multistart.csv"), ms_rows,
          ["mode", "rpm", "worst_node", "builtin_torque", "multistart_best", "n_improvements",
           "n_starts", "peak_curr", "curr_used_pct", "verdict", "seconds"])
print("saved worst_node_multistart.csv")

In [ ]:
PROBE_CASES = [("healthy", 0.0), ("healthy", 1500.0)]
PROBE_TIMEOUT = 120.0
PROBE_MARGINS = (1.0, 0.999, 0.995, 0.99, 0.95, 0.9)

probe_rows = []
for mode, rpm in PROBE_CASES:
    opt = make_opt(mode)
    omega = rpm / CONST_MECH_SPEED
    maps = opt._precompute_maps(omega)
    t0 = time.perf_counter()
    t_max_r0, _, _ = r0_probe(opt, omega, maps)
    t_ceiling = time.perf_counter() - t0
    print(f"=== {mode} @ {rpm} RPM: ceiling {t_max_r0:.3f} Nm (probe {t_ceiling:.1f}s) ===", flush=True)
    if not np.isfinite(t_max_r0) or t_max_r0 <= 0:
        print("  ceiling <= 0, backoff loop never runs", flush=True)
        continue
    total = t_ceiling
    for margin in PROBE_MARGINS:
        t0 = time.perf_counter()
        sol, err = call_with_timeout(PROBE_TIMEOUT, opt.minimize_current, t_max_r0 * margin, omega)
        dt = time.perf_counter() - t0
        total += dt
        ok = sol is not None and sol.success
        probe_rows.append({"mode": mode, "rpm": rpm, "margin": margin, "target": t_max_r0 * margin,
                           "converged": ok, "error": err or "", "seconds": dt,
                           "cumulative_seconds": total})
        print(f"  margin {margin:.3f}  target {t_max_r0 * margin:6.3f} Nm  -> "
              f"{'converged' if ok else (err or 'failed')}  ({dt:.1f}s, cumulative {total:.1f}s)", flush=True)
        if ok:
            break

write_csv(os.path.join(OUT_DIR, "probe_cost.csv"), probe_rows,
          ["mode", "rpm", "margin", "target", "converged", "error", "seconds", "cumulative_seconds"])
print("\nsaved probe_cost.csv")
print("Every full timeout here is pure waste paid on EVERY speed row of a sweep.")

In [ ]:
SWEEP_MODE, SWEEP_RPM = "1-fault", 1200.0
SWEEP_FRACS = [0.2, 0.35, 0.5, 0.65, 0.8]

if SWEEP_MODE is not None:
    ceiling_rows, ceiling_hist = [], []
    CEILING_CSV = os.path.join(OUT_DIR, f"torque_sweep_{SWEEP_MODE}_{SWEEP_RPM:.0f}rpm.csv")
    print(f"=== {SWEEP_MODE} @ {SWEEP_RPM} RPM: torque-fraction sweep ===", flush=True)
    for frac in SWEEP_FRACS:
        row, hrows, _ = analyze_point(SWEEP_MODE, SWEEP_RPM, frac=frac, timeout=POINT_TIMEOUT)
        ceiling_rows.append(row)
        ceiling_hist.extend(hrows)
        write_csv(CEILING_CSV, ceiling_rows, DETAIL_FIELDS)

    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    good = [(r["torque"], max(r["final_resid"], 1e-6)) for r in ceiling_rows
            if r.get("verified_feasible") and np.isfinite(r.get("final_resid", np.nan))]
    bad = [(r["torque"], max(r["final_resid"], 1e-6)) for r in ceiling_rows
           if not r.get("verified_feasible") and np.isfinite(r.get("final_resid", np.nan))]
    if good:
        ax.scatter(*zip(*good), c="tab:green", s=90, marker="o", edgecolors="k", label="feasible", zorder=3)
    if bad:
        ax.scatter(*zip(*bad), c="tab:red", s=90, marker="X", edgecolors="k", label="missed tolerance", zorder=3)
    ax.axhline(TOL, color="k", ls="--", lw=1.2, label=f"TOL = {TOL:g} V")
    ax.set_yscale("log")
    ax.set_xlabel("target torque [Nm]"); ax.set_ylabel("final max exact residual [V]")
    ax.set_title(f"{SWEEP_MODE} @ {SWEEP_RPM:.0f} RPM: residual reached vs requested torque")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
    plt.tight_layout()
    out = os.path.join(OUT_DIR, f"torque_sweep_{SWEEP_MODE}_{SWEEP_RPM:.0f}rpm.png")
    fig.savefig(out, dpi=110)
    plt.close(fig)
    print(f"\nsaved {CEILING_CSV} and {out}")
    print("If low fractions fail too, the ceiling is not the cause -- the solver")
    print("stalls at this operating point regardless of the torque requested.")

In [ ]:
WAVE_DIR = os.path.join(OUT_DIR, "waveforms")
os.makedirs(WAVE_DIR, exist_ok=True)


def segment_label(n_curr, n_volt):
    roman = {0: "", 1: "I", 2: "II"}
    return f"MTPA_{n_curr}" if n_volt == 0 else f"FW_{roman[n_volt]},{n_curr}"


def fine_idx_grid(fwd, theta_grid):
    n_theta = fwd.vec_theta.size - 1
    return np.round(theta_grid / (2 * np.pi) * n_theta).astype(int) % n_theta


def reconstruct_phase_currents(fwd, theta_grid, sched):
    idx_grid = fine_idx_grid(fwd, theta_grid)
    n_surv = fwd.phase_map_at_theta(0).shape[0]
    out = np.full((n_surv, len(idx_grid)), np.nan)
    for k, ti in enumerate(idx_grid):
        if np.all(np.isfinite(sched[k])):
            out[:, k] = fwd.phase_map_at_theta(int(ti)) @ sched[k]
    return out


def reconstruct_phase_voltages(fwd, theta_grid, sched, omega):
    idx_grid = fine_idx_grid(fwd, theta_grid)
    kept = list(fwd.fault._kept)
    out = np.full((len(kept), len(idx_grid)), np.nan)
    for k, ti in enumerate(idx_grid):
        i_n = sched[k]
        if np.all(np.isfinite(i_n)):
            gU, _, bV = fwd.volt_map_at_theta(int(ti), omega, i_n)
            out[:, k] = (gU @ i_n + bV)[kept]
    return out


def alpha_beta_xy(theta_grid, sched):
    c1, s1 = np.cos(theta_grid), np.sin(theta_grid)
    c3, s3 = np.cos(3 * theta_grid), np.sin(3 * theta_grid)
    id1, iq1, id3, iq3 = sched[:, 0], sched[:, 1], sched[:, 2], sched[:, 3]
    i_alpha = c1 * id1 - s1 * iq1
    i_beta = s1 * id1 + c1 * iq1
    i_x = c3 * id3 - s3 * iq3
    i_y = s3 * id3 + c3 * iq3
    return i_alpha, i_beta, i_x, i_y


def save_setpoint_plot(fwd, sol, torque, rpm, omega, save_path, mode):
    """Same 9-subplot figure as pointwise_waveform_diagnostics.ipynb's
    plot_point, but saved to disk instead of shown -- for use in an
    unattended overnight sweep."""
    sched = sol.curr_dq
    theta_grid = sol.diagnostics["theta_grid"]
    theta = np.rad2deg(theta_grid)

    iph = reconstruct_phase_currents(fwd, theta_grid, sched)
    vph = reconstruct_phase_voltages(fwd, theta_grid, sched, omega)
    i_alpha, i_beta, i_x, i_y = alpha_beta_xy(theta_grid, sched)

    n_curr = count_peaks_at_limit(np.concatenate([iph, iph[:, :1]], axis=1), CURR_MAX)
    n_volt = count_peaks_at_limit(np.concatenate([vph, vph[:, :1]], axis=1), VOLT_MAX)
    seg_text = segment_label(n_curr, n_volt)

    kept = list(fwd.fault._kept)
    letters = [PHASE_LETTERS[p] for p in kept]

    norm_pw = np.sum(sched ** 2, axis=1)
    cu_pw = np.nansum(iph ** 2, axis=0)
    torque_pw = np.array([
        fwd.drive.torque(omega, sched[k]) if np.all(np.isfinite(sched[k])) else np.nan
        for k in range(sched.shape[0])
    ])

    have_fft = np.all(np.isfinite(iph))
    if have_fft:
        iph_fft = np.abs(np.fft.rfft(iph, axis=1)) / iph.shape[1]
        harmonics = np.arange(iph_fft.shape[1])

    fig, ax = plt.subplots(3, 3, figsize=(18, 13))
    fig.suptitle(f"{mode}  [{seg_text}]   T*={torque:.2f} Nm,  {rpm:.1f} RPM", fontsize=13)

    dq_lbl = [r"$i_{sd}^1$", r"$i_{sq}^1$", r"$i_{sd}^3$", r"$i_{sq}^3$"]
    dq_col = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
    a = ax[0, 0]
    for j in range(fwd.drive.dim):
        a.plot(theta, sched[:, j], color=dq_col[j], label=dq_lbl[j])
    a.set_title("DQ setpoints")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("DQ current [A]")
    a.legend(fontsize=8)

    a = ax[0, 1]
    for p, lt in enumerate(letters):
        a.plot(theta, iph[p], label=f"$i_{{{lt}}}$")
    a.axhline(CURR_MAX, color="gray", ls=":"); a.axhline(-CURR_MAX, color="gray", ls=":")
    a.set_title("Phase currents")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("phase current [A]")
    a.legend(fontsize=8)

    a = ax[0, 2]
    for p, lt in enumerate(letters):
        a.plot(theta, vph[p], label=f"$v_{{{lt}}}$")
    a.axhline(VOLT_MAX, color="gray", ls=":"); a.axhline(-VOLT_MAX, color="gray", ls=":")
    a.set_title("Phase voltages")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("phase voltage [V]")
    a.legend(fontsize=8)

    a = ax[1, 0]
    a.plot(theta, cu_pw, color="tab:orange", label=f"mean={np.nanmean(cu_pw):.1f}")
    a.set_title(r"Copper-loss density $\sum_k i_k^2(\theta)$")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel(r"$\sum_k i_k^2$ [A$^2$]")
    a.legend(fontsize=8)

    a = ax[1, 1]
    a.plot(theta, norm_pw, color="tab:orange", label=r"$\|i_s(\theta)\|^2$")
    a.set_title(r"DQ norm $\|i_s\|^2$")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel(r"$\|i_s\|^2$ [A$^2$]")
    a.legend(fontsize=8)

    a = ax[1, 2]
    a.plot(i_alpha, i_beta, color="tab:purple")
    a.set_title(r"$\alpha\beta$ current")
    a.set_xlabel(r"$i_\alpha$ [A]"); a.set_ylabel(r"$i_\beta$ [A]")
    a.set_aspect("equal")

    a = ax[2, 0]
    a.plot(i_x, i_y, color="tab:brown")
    a.set_title("xy current")
    a.set_xlabel(r"$i_x$ [A]"); a.set_ylabel(r"$i_y$ [A]")
    a.set_aspect("equal")

    a = ax[2, 1]
    a.plot(theta, torque_pw, color="tab:red", label=f"mean={np.nanmean(torque_pw):.3f}")
    a.axhline(torque, color="gray", ls=":", label=f"T*={torque:.2f}")
    a.set_title("Achieved torque (per-node constraint check)")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("torque [Nm]")
    a.legend(fontsize=8)

    a = ax[2, 2]
    if have_fft:
        width = 0.8 / len(letters)
        for p, lt in enumerate(letters):
            a.bar(harmonics + p * width, iph_fft[p], width=width, label=f"$i_{{{lt}}}$")
        a.set_xlim(-0.5, min(15, harmonics[-1]) + 0.5)
        a.set_title("Phase current spectrum (electrical harmonics)")
        a.set_xlabel("harmonic order"); a.set_ylabel("amplitude [A]")
        a.legend(fontsize=8)
    else:
        a.axis("off")
        a.text(0.5, 0.5, "spectrum skipped:\nsome nodes infeasible/NaN",
               ha="center", va="center", transform=a.transAxes)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(save_path, dpi=110)
    plt.close(fig)


def plot_setpoint_waveforms(mode, rpm, torque, timeout=900.0, tag=""):
    """Solve one setpoint and draw the sweep's figure for it whether or not it
    converged. Saves PNG + per-node CSV; returns the summary row."""
    fwd = FWD_BY_MODE[mode]
    opt = make_opt(mode)
    omega = rpm / CONST_MECH_SPEED
    maps = opt._precompute_maps(omega)
    idx_grid = maps["idx_grid"]

    t0 = time.perf_counter()
    sol, err = call_with_timeout(timeout, opt.minimize_current, torque, omega)
    secs = time.perf_counter() - t0
    if sol is None:
        print(f"  {mode} {rpm:.0f} RPM T={torque:.3f}: {err} -- nothing to plot")
        return {"mode": mode, "rpm": rpm, "torque": torque, "outcome": err or "no solution",
                "verified_feasible": False, "seconds": secs}

    X = sol.curr_dq
    node_ok = np.all(np.isfinite(X), axis=1)
    if node_ok.all():
        r, _ = opt.voltage_residuals_exact(omega, X, idx_grid)
        final_resid = float(np.max(r))
    else:
        r = np.full(N_THETA_GRID, np.nan)
        final_resid = np.nan
    feasible = bool(np.isfinite(final_resid) and final_resid <= TOL)
    miss = final_resid - TOL if np.isfinite(final_resid) else np.nan
    verdict = "FEASIBLE" if feasible else "INFEASIBLE"

    stem = f"{tag}{mode}_T{torque:07.3f}_RPM{rpm:06.0f}_{'OK' if feasible else 'FAIL'}"
    png = os.path.join(WAVE_DIR, stem + ".png")
    try:
        save_setpoint_plot(fwd, sol, torque, rpm, omega, png, mode)
        plotted = True
    except Exception as exc:
        print(f"    plot failed: {exc!r}")
        plotted = False

    theta_grid = sol.diagnostics["theta_grid"]
    iph = reconstruct_phase_currents(fwd, theta_grid, X)
    vph = reconstruct_phase_voltages(fwd, theta_grid, X, omega)
    kept = list(fwd.fault._kept)
    letters = [PHASE_LETTERS[p] for p in kept]
    wave_rows = [{"node": n, "theta_deg": float(np.rad2deg(theta_grid[n])), "resid": r[n],
                  "i_sd1": X[n, 0], "i_sq1": X[n, 1], "i_sd3": X[n, 2], "i_sq3": X[n, 3],
                  **{f"i_{letters[p]}": iph[p, n] for p in range(len(kept))},
                  **{f"v_static_{letters[p]}": vph[p, n] for p in range(len(kept))}}
                 for n in range(N_THETA_GRID)]
    write_csv(os.path.join(WAVE_DIR, stem + ".csv"), wave_rows, list(wave_rows[0]))

    hist = sol.diagnostics.get("history", [])
    print(f"  [{verdict:10s}] {mode:10s} {rpm:6.0f} RPM T={torque:6.3f}  "
          f"exact resid {final_resid:+8.4f} V  miss {miss:+.4f} V  "
          f"iters={sum(1 for h in hist if isinstance(h.get('iter'), int) and h['iter'] >= 0)}  ({secs:.0f}s)")
    return {"mode": mode, "rpm": rpm, "torque": torque, "final_resid": final_resid,
            "miss": miss, "resid_pct_of_limit": final_resid / VOLT_MAX * 100.0,
            "verified_feasible": feasible, "success": bool(sol.success),
            "outcome": sol.diagnostics.get("fail_reason", "") or (hist[-1].get("action", "") if hist else ""),
            "n_nan_nodes": int((~node_ok).sum()), "png": os.path.basename(png) if plotted else "",
            "seconds": secs}

In [ ]:
CENTRE_MODE, CENTRE_RPM = "1-fault", 1200.0
CENTRE_FRAC = 0.5
TORQUE_SCALES = [0.6, 0.8, 1.0, 1.2]
RPM_OFFSETS = [0.0]

_opt = make_opt(CENTRE_MODE)
_maps = _opt._precompute_maps(CENTRE_RPM / CONST_MECH_SPEED)
_ceiling, _, _ = r0_probe(_opt, CENTRE_RPM / CONST_MECH_SPEED, _maps)
CENTRE_TORQUE = CENTRE_FRAC * _ceiling
print(f"centre: {CENTRE_MODE} @ {CENTRE_RPM:.0f} RPM, R0 ceiling {_ceiling:.3f} Nm, "
      f"centre torque {CENTRE_TORQUE:.3f} Nm ({CENTRE_FRAC:.0%})")
print(f"{len(TORQUE_SCALES) * len(RPM_OFFSETS)} points to solve\n", flush=True)

neigh_rows = []
NEIGH_CSV = os.path.join(OUT_DIR, "neighbourhood.csv")
for d_rpm in RPM_OFFSETS:
    for scale in TORQUE_SCALES:
        row = plot_setpoint_waveforms(CENTRE_MODE, CENTRE_RPM + d_rpm,
                                      CENTRE_TORQUE * scale, tag="nb_")
        neigh_rows.append(row)
        write_csv(NEIGH_CSV, neigh_rows,
                  ["mode", "rpm", "torque", "final_resid", "miss", "resid_pct_of_limit",
                   "verified_feasible", "success", "outcome", "n_nan_nodes", "png", "seconds"])

print(f"\nsaved {NEIGH_CSV} and {len(neigh_rows)} figures in {WAVE_DIR}")
ok = [r for r in neigh_rows if r.get("verified_feasible")]
print(f"{len(ok)}/{len(neigh_rows)} feasible; every point was plotted regardless")

In [ ]:
for root in (OUT_DIR, os.path.join(OUT_DIR, "waveforms")):
    if not os.path.isdir(root):
        continue
    print(f"\n{os.path.abspath(root)}")
    print(f"  {'file':56s} {'size':>9s}  {'modified':>20s}")
    print("  " + "-" * 88)
    for name in sorted(os.listdir(root)):
        p = os.path.join(root, name)
        if os.path.isdir(p):
            continue
        st = os.stat(p)
        print(f"  {name:56s} {st.st_size:9d}  "
              f"{time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(st.st_mtime)):>20s}")